# Electron g-2 Precision — v2: Lattice-based 600-cell Monte Carlo

Upgraded from scalar approximation to explicit 600-cell walks with phase coherence.

Features:
- Exact 120-vertex coordinates (golden ratio based)
- Approximate adjacency (12 neighbors per vertex)
- Short geodesic walks with accumulated phases
- Coherent interference (complex amplitudes)
- Batching to handle 500M+ paths without OOM

Goal: stronger destructive interference → δμ_e suppression toward 10^{-12} or better.

In [ ]:
import cupy as cp
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import time

# ====================== PARAMETERS ======================
N_paths_total = 500_000_000     # total paths (will be split into batches)
BATCH_SIZE    = 50_000_000      # adjust down if OOM (try 25M–75M)
n_batches     = (N_paths_total + BATCH_SIZE - 1) // BATCH_SIZE

WALK_LENGTH   = 10              # steps per path (short geodesics)
phase_scale   = 6.0             # tune: larger → more oscillation/cancellation
fbs_power     = 2.0             # FBS grading exponent

C = 9.6e-7                      # overall constant
alpha = 1 / 137.035999084

print(f"Starting v2 lattice simulation: {N_paths_total:,} total paths ({BATCH_SIZE:,} per batch)")

In [ ]:
# ====================== 600-CELL VERTICES (120 points) ======================
phi = (1 + np.sqrt(5)) / 2
inv_phi = 1 / phi

def generate_600cell_vertices():
    coords = []
    # 16: even perms of (±1/2, ±1/2, ±1/2, ±1/2)
    for signs in np.array(np.meshgrid(*[[-0.5,0.5]]*4)).T.reshape(-1,4):
        coords.append(signs)
    # 8: perms of (±1, 0, 0, 0)
    for i in range(4):
        for s in [-1,1]:
            v = np.zeros(4)
            v[i] = s
            coords.append(v)
    # 96: even perms of (±1/2 * φ, ±1/2, ±1/2 * φ⁻¹, 0) and sign variants
    base = [0.5*phi, 0.5, 0.5*inv_phi, 0]
    for perm in [[0,1,2,3],[0,1,3,2],[0,2,1,3],[0,2,3,1],[0,3,1,2],[0,3,2,1]]:
        for signs in [[1,1,1],[1,1,-1],[1,-1,1],[-1,1,1]]:
            v = np.zeros(4)
            v[perm[0]] = signs[0] * base[0]
            v[perm[1]] = signs[1] * base[1]
            v[perm[2]] = signs[2] * base[2]
            coords.append(v)
    verts = np.unique(np.round(coords, decimals=10), axis=0)  # remove near-duplicates
    verts = verts / np.linalg.norm(verts, axis=1)[:, None]   # unit norm
    return cp.asarray(verts, dtype=cp.float32)

vertices = generate_600cell_vertices()
N_VERT = vertices.shape[0]
print(f"Generated {N_VERT} vertices (should be 120)")

In [ ]:
# ====================== ADJACENCY (approximate, degree ~12) ======================
dists = cp.linalg.norm(vertices[:, None] - vertices[None, :], axis=-1)
edge_threshold = cp.sort(dists.flatten())[720*2 + 1]  # 720 edges → threshold after smallest 1440 directed
adj_mask = (dists < edge_threshold + 1e-6) & (dists > 1e-6)
neighbors = [cp.argwhere(adj_mask[i])[:,0] for i in range(N_VERT)]
degrees = [len(n) for n in neighbors]
print(f"Average degree: {np.mean(degrees):.2f} (ideal 12)")
print(f"Min/Max degree: {min(degrees)} / {max(degrees)}")

In [ ]:
# ====================== MAIN SIMULATION ======================
raw_amplitudes = cp.zeros(3, dtype=cp.complex64)  # accumulators for e/h/q channels

start_time = time.time()
for batch_idx in tqdm(range(n_batches), desc="Batches"):
    batch_size = min(BATCH_SIZE, N_paths_total - batch_idx * BATCH_SIZE)

    # Start at random vertices
    current = cp.random.randint(0, N_VERT, batch_size)

    phase = cp.zeros(batch_size, dtype=cp.complex64)
    fbs_grade = cp.ones(batch_size, dtype=cp.float32)

    for step in range(WALK_LENGTH):
        # Pick random neighbor for each path
        neigh_choices = cp.array([cp.random.choice(n, size=1)[0] for n in neighbors])[current]

        # Simple edge length (chordal distance)
        edge_len = cp.linalg.norm(vertices[current] - vertices[neigh_choices], axis=1)

        # Accumulate phase
        phase += 1j * phase_scale * edge_len * fbs_grade

        # Update FBS grading (stronger suppression on longer paths)
        fbs_grade *= 1.0 / (edge_len ** fbs_power + 0.1)

        current = neigh_choices

    # Interference: complex amplitude per path
    amp = cp.exp(phase) * fbs_grade

    # Channel projection (placeholder phases — tune these!)
    e_amp = amp * cp.exp(1j * 0.0)
    h_amp = amp * cp.exp(1j * 1.2)
    q_amp = amp * cp.exp(1j * 2.4)

    raw_amplitudes[0] += cp.sum(e_amp)
    raw_amplitudes[1] += cp.sum(h_amp)
    raw_amplitudes[2] += cp.sum(q_amp)

# Normalize
interference = raw_amplitudes / N_paths_total
abs_interf = cp.abs(interference)

# Fractions from |amplitudes|
total_mag = cp.sum(abs_interf)
eDP = float(abs_interf[0] / total_mag)
hDP = float(abs_interf[1] / total_mag)
qDP = float(abs_interf[2] / total_mag)

print(f"eDP: {eDP:.6f}")
print(f"hDP: {hDP:.6f}")
print(f"qDP: {qDP:.6f}")

# Suppression & δμ_e
S = alpha / (2 * np.pi) * cp.mean(abs_interf)  # stronger via coherence
mixing_sum = qDP + 0.7 * hDP
delta_mu_e = C * mixing_sum * S

print(f"\nRaw mixing boost approx: {C * mixing_sum:.2e}")
print(f"Coherence suppression S: {S:.3e}")
print(f"Electron δμ_e (v2 interference): {delta_mu_e:.3e}")

print(f"\nCompleted in {time.time() - start_time:.1f} seconds")

## Tuning suggestions
- Increase `phase_scale` (8–15) for more destructive interference.
- Try `WALK_LENGTH = 12–16` if memory allows.
- Add channel-specific phase offsets.
- Next: proper multi-generation loops (muon/tau mass scaling).
- Save improved results and push to GitHub!